In [6]:
!pip -q install -U mlflow dagshub

import os
import mlflow
import dagshub
import pandas as pd
import numpy as np

print("MLflow version:", mlflow.__version__)

MLflow version: 3.14.0


In [8]:
import dagshub

token = dagshub.auth.get_token()

print("Token received successfully!")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e3bead80-320b-4810-855d-188105484fa2&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e4a459dfe824ac91914fd5bcaea891a90fd0e8eeaab4b980e7639feb36c2b02e




Accessing as lkhar21

Token received successfully!


In [17]:
from google.colab import userdata

token = userdata.get("DAGSHUB_TOKEN")

print("Token loaded successfully:", bool(token))

Token loaded successfully: True


In [19]:
import os
import mlflow
from google.colab import userdata

token = userdata.get("DAGSHUB_TOKEN").strip()

# ძველი/არასწორი ავტორიზაციის ცვლადების გასუფთავება
os.environ.pop("MLFLOW_TRACKING_USERNAME", None)
os.environ.pop("MLFLOW_TRACKING_PASSWORD", None)

# DagsHub-ის მხარდაჭერილი ვარიანტი:
# token გამოიყენება username-ის სახით, password აღარ გვჭირდება
os.environ["MLFLOW_TRACKING_USERNAME"] = token

TRACKING_URI = (
    "https://dagshub.com/tsarc21/"
    "Walmart-Recruiting---Store-Sales-Forecasting.mlflow"
)

mlflow.set_tracking_uri(TRACKING_URI)

MODEL_URI = "models:/Walmart_Optimized_Seasonal_Lag/1"

loaded_model = mlflow.pyfunc.load_model(MODEL_URI)

print("✅ Model loaded successfully!")
print("Model URI:", MODEL_URI)

✅ Model loaded successfully!
Model URI: models:/Walmart_Optimized_Seasonal_Lag/1


In [21]:
from google.colab import files
import os
import shutil

uploaded = files.upload()   # აქ ატვირთე მხოლოდ kaggle.json

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod(
    "/root/.kaggle/kaggle.json",
    0o600
)

print("✅ Kaggle API configured successfully!")

Saving kaggle.json to kaggle (1).json
✅ Kaggle API configured successfully!


In [22]:
import os
import zipfile

COMPETITION = "walmart-recruiting-store-sales-forecasting"

os.makedirs("/content/data", exist_ok=True)

!kaggle competitions download \
    -c {COMPETITION} \
    -p /content/data

competition_zip = (
    f"/content/data/{COMPETITION}.zip"
)

with zipfile.ZipFile(competition_zip, "r") as zip_ref:
    zip_ref.extractall("/content/data")

# კონკურსის შიგნით არსებული ცალკეული ZIP ფაილების გახსნა
for filename in os.listdir("/content/data"):
    filepath = os.path.join("/content/data", filename)

    if filename.endswith(".zip") and filepath != competition_zip:
        with zipfile.ZipFile(filepath, "r") as zip_ref:
            zip_ref.extractall("/content/data")

print("✅ Files downloaded and extracted:")
print(os.listdir("/content/data"))

100% 2.70M/2.70M [00:00<00:00, 193MB/s]

✅ Files downloaded and extracted:
['test.csv', 'test.csv.zip', 'stores.csv', 'sampleSubmission.csv.zip', 'train.csv', 'train.csv.zip', 'walmart-recruiting-store-sales-forecasting.zip', 'sampleSubmission.csv', 'features.csv', 'features.csv.zip']


In [23]:
import pandas as pd

test = pd.read_csv("/content/data/test.csv")
sample_submission = pd.read_csv("/content/data/sampleSubmission.csv")

test["Date"] = pd.to_datetime(test["Date"])

model_input = test[
    ["Store", "Dept", "Date", "IsHoliday"]
].copy()

print("Test shape:", test.shape)
print("Model input shape:", model_input.shape)
print("Sample submission shape:", sample_submission.shape)

display(model_input.head())
display(sample_submission.head())

Test shape: (115064, 4)
Model input shape: (115064, 4)
Sample submission shape: (115064, 2)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


,Id,Weekly_Sales
0,1_1_2012-11-02,0
1,1_1_2012-11-09,0
2,1_1_2012-11-16,0
3,1_1_2012-11-23,0
4,1_1_2012-11-30,0


In [24]:
predictions = loaded_model.predict(model_input)

if isinstance(predictions, pd.DataFrame):
    weekly_sales = predictions["Weekly_Sales"].to_numpy()
else:
    weekly_sales = predictions

if len(weekly_sales) != len(sample_submission):
    raise ValueError(
        f"Prediction count mismatch: "
        f"{len(weekly_sales)} predictions, "
        f"{len(sample_submission)} submission rows"
    )

submission = sample_submission.copy()
submission["Weekly_Sales"] = weekly_sales

submission.to_csv(
    "/content/submission.csv",
    index=False
)

print("✅ submission.csv created successfully!")
print("Submission shape:", submission.shape)
print("Missing predictions:", submission["Weekly_Sales"].isna().sum())

display(submission.head(10))

✅ submission.csv created successfully!
Submission shape: (115064, 2)
Missing predictions: 0


,Id,Weekly_Sales
0,1_1_2012-11-02,33570.2160
1,1_1_2012-11-09,24060.8940
2,1_1_2012-11-16,19332.4980
3,1_1_2012-11-23,20977.3560
4,1_1_2012-11-30,25800.4160
5,1_1_2012-12-07,33796.2345
6,1_1_2012-12-14,42859.3965
7,1_1_2012-12-21,41847.2460
8,1_1_2012-12-28,27360.1350
9,1_1_2013-01-04,18328.8295


In [25]:
import os
import pandas as pd

SUBMISSION_PATH = "/content/submission.csv"

if not os.path.exists(SUBMISSION_PATH):
    raise FileNotFoundError("submission.csv ვერ მოიძებნა.")

submission_check = pd.read_csv(SUBMISSION_PATH)

expected_columns = ["Id", "Weekly_Sales"]

if submission_check.columns.tolist() != expected_columns:
    raise ValueError(
        f"არასწორი სვეტები: {submission_check.columns.tolist()}\n"
        f"საჭიროა: {expected_columns}"
    )

if submission_check["Weekly_Sales"].isna().any():
    raise ValueError("Weekly_Sales სვეტში NaN მნიშვნელობებია.")

print("✅ Submission შემოწმებულია")
print("Shape:", submission_check.shape)
display(submission_check.head())

✅ Submission შემოწმებულია
Shape: (115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,33570.216
1,1_1_2012-11-09,24060.894
2,1_1_2012-11-16,19332.498
3,1_1_2012-11-23,20977.356
4,1_1_2012-11-30,25800.416


In [26]:
COMPETITION = "walmart-recruiting-store-sales-forecasting"
MESSAGE = "Optimized Seasonal Lag 51-52-53 from MLflow Model Registry"

!kaggle competitions submit \
    -c {COMPETITION} \
    -f /content/submission.csv \
    -m "{MESSAGE}"

100% 3.29M/3.29M [00:00<00:00, 13.7MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting